In [1]:
import os
import json
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer

from pprint import pprint

In [2]:
print("=" * 50)

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

print("=" * 50)

Torch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA Version: 12.4


## Configuration

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

TRAIN_PATH = "../datasets/chat/train_chat.json"

OUTPUT_DIR = "../models/qwen-med-sft"

MAX_SEQ_LENGTH = 1024

BATCH_SIZE = 1

GRAD_ACCUM = 2

LEARNING_RATE = 2e-4

EPOCHS = 3

SEED = 42

In [4]:
with open(TRAIN_PATH, "r") as f:
    train_data = json.load(f)

print(len(train_data))

10178


In [6]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)

train_dataset = train_dataset.shuffle(seed=42)

train_dataset = train_dataset.select(range(2000))

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [9]:
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False
)

print(text)

<|im_start|>system
You are an expert medical AI assistant. Carefully analyze the clinical question and select the most appropriate answer.<|im_end|>
<|im_start|>user
Clinical Question:

A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:

Options:
A. Pallor, cyanosis, and erythema of the hands
B. Blanching vascular abnormalities
C. Hypercoagulable state
D. Heartburn and regurgitation

Select the best answer.<|im_end|>
<|im_start|>assistant
The correct answer is C.
Hypercoagulable state<|im_end|>



In [10]:
tokens = tokenizer(
    text,
    return_tensors="pt"
)

print(tokens.input_ids.shape)

torch.Size([1, 154])


In [11]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True

)

In [12]:
model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True
)

In [13]:
model = prepare_model_for_kbit_training(model)

In [14]:
model.gradient_checkpointing_enable()

In [15]:
# Lora Config: 


peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[

        "q_proj",

        "k_proj",

        "v_proj",

        "o_proj",

        "gate_proj",

        "up_proj",

        "down_proj"

    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"

)

In [16]:
model = get_peft_model(
    model,
    peft_config
)

In [17]:
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [18]:
formatted_text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False
)

print(formatted_text)

<|im_start|>system
You are an expert medical AI assistant. Carefully analyze the clinical question and select the most appropriate answer.<|im_end|>
<|im_start|>user
Clinical Question:

A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:

Options:
A. Pallor, cyanosis, and erythema of the hands
B. Blanching vascular abnormalities
C. Hypercoagulable state
D. Heartburn and regurgitation

Select the best answer.<|im_end|>
<|im_start|>assistant
The correct answer is C.
Hypercoagulable state<|im_end|>



In [19]:
train_dataset[0]

{'id': 'train_2622',
 'messages': [{'content': 'You are an expert medical AI assistant. Carefully analyze the clinical question and select the most appropriate answer.',
   'role': 'system'},
  {'content': 'Clinical Question:\n\nA 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:\n\nOptions:\nA. Pallor, cyanosis, and erythema of the hands\nB. Blanching vascular abnormalities\nC. Hypercoagulable state\nD. Heartburn and regurgitation\n\nSelect the best answer.',
   'role': 'user'},
  {'content': 'The correct answer is C.\nHypercoagulable state',
   'role': 'assistant'}]}

In [20]:
def formatting_func(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = train_dataset.map(formatting_func)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [21]:
print(train_dataset.column_names)

['id', 'messages', 'text']


In [22]:
print(train_dataset[0]["text"])

<|im_start|>system
You are an expert medical AI assistant. Carefully analyze the clinical question and select the most appropriate answer.<|im_end|>
<|im_start|>user
Clinical Question:

A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:

Options:
A. Pallor, cyanosis, and erythema of the hands
B. Blanching vascular abnormalities
C. Hypercoagulable state
D. Heartburn and regurgitation

Select the best answer.<|im_end|>
<|im_start|>assistant
The correct answer is C.
Hypercoagulable state<|im_end|>



In [23]:
from trl import SFTConfig

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    report_to="none",

    bf16=torch.cuda.is_available(),

    fp16=False,

    max_length=1024,

    packing=False,

    seed=SEED,
)

In [24]:
trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.000100
20,1.661800
30,1.560000
40,1.514700
50,1.606200
60,1.558700
70,1.565400
80,1.498300
90,1.583800
100,1.461800


TrainOutput(global_step=3000, training_loss=1.2701519492467244, metrics={'train_runtime': 2535.2172, 'train_samples_per_second': 2.367, 'train_steps_per_second': 1.183, 'total_flos': 3485682473852160.0, 'train_loss': 1.2701519492467244, 'epoch': 3.0})

In [26]:
import trl
print(trl.__version__)

1.7.1


In [27]:
trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

('../models/qwen-med-sft/tokenizer_config.json',
 '../models/qwen-med-sft/special_tokens_map.json',
 '../models/qwen-med-sft/chat_template.jinja',
 '../models/qwen-med-sft/vocab.json',
 '../models/qwen-med-sft/merges.txt',
 '../models/qwen-med-sft/added_tokens.json',
 '../models/qwen-med-sft/tokenizer.json')

### Testing

In [80]:
question = """
A patient presents with Headache & Vomitting.
"""

# question = """
# Clinical Question:

# A 55-year-old man presents with crushing substernal chest pain radiating to his left arm. ECG demonstrates ST-segment elevation in leads II, III, and aVF.

# Options:
# A. Gastroesophageal reflux disease
# B. Acute myocardial infarction
# C. Costochondritis
# D. Panic attack

# Select the best answer.
# """

messages = [

    {
        "role": "system",
        "content": "You are an expert medical AI assistant."
    },

    {
        "role": "user",
        "content": question
    }
]

In [81]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():

    outputs = model.generate(

        **inputs,

        max_new_tokens=128,

        do_sample=False,

        temperature=0.2,

        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,

    )

In [ ]:
generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

The patient is most likely suffering from which of the following conditions?
 

Options:
A. Migraine headache
B. Episodic migraine
C. Tension headache
D. Cluster headache

Select the best answer.andest
andest
The correct answer is A.
Migraine headacheandest
andest
The correct answer is A.
Migraine headacheandest
andest
The correct answer is A.
Migraine headacheandest
andest
The correct answer is A.
Migraine headacheandest
andest
The correct answer is A.
Migraine headacheandest
andest
The correct answer is A.
Migr


: 